# Imputacion valores NAN y OUTLIERS segun rango temporal con metodo por Estacion y metodo General

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 2: Imputación de valores NaN según duración del gap
y dos estrategias: por estación individual y global (todas las estaciones).

Requisitos:
- pandas, numpy, scikit-learn (para KNNImputer)
- Los archivos de entrada son los generados en el Código 1 (*_outliers.csv)
  que contienen la columna 'O3_for_impute' con los NaN de errores de sensor.
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import KNNImputer

# ============================================================================
# PARÁMETROS CONFIGURABLES
# ============================================================================

# Carpeta donde están los archivos con outliers (salida del Código 1)
INPUT_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")

# Carpetas de salida para los datos imputados
OUTPUT_BY_STATION = os.path.join(INPUT_DIR, "imputed_by_station")
OUTPUT_GLOBAL = os.path.join(INPUT_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_STATION, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

# Umbrales para los métodos de imputación (en horas)
SHORT_GAP_MAX = 6          # ≤ 6h → interpolación lineal
MEDIUM_GAP_MAX = 72        # >6h y <72h → estacional por hora
LONG_GAP_MIN = 72          # ≥72h → multivariante (KNN)

# Método para la imputación estacional: 'mean' o 'median'
SEASONAL_STAT = 'mean'

# Columnas numéricas a imputar (incluye O3_for_impute, que será la O3 final)
NUM_COLS = [
    'NO', 'NO2', 'NOx', 'O3_for_impute',   # O3 se imputa desde esta columna
    'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Dist.', 'Angulo'
]

# Columnas que se conservan pero no se imputan (categóricas, flags, etc.)
# (Se mantienen intactas en el output)

# Para KNNImputer (multivariante)
KNN_NEIGHBORS = 5

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def load_station_data(filename):
    """Carga un archivo CSV de una estación (con índice datetime)."""
    df = pd.read_csv(filename, index_col=0, parse_dates=True)
    # Asegurar que el índice es datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df

def get_gap_lengths(series):
    """
    Devuelve un diccionario con los grupos de NaN y su longitud.
    Cada grupo es un objeto Index con las posiciones del gap.
    """
    mask = series.isna()
    # Identificar grupos consecutivos de True (NaN)
    groups = mask.ne(mask.shift()).cumsum()
    gap_groups = {}
    for g in groups[mask].unique():
        idx = groups[groups == g].index
        gap_groups[tuple(idx)] = len(idx)   # guardamos el índice como tupla para poder iterar
    return gap_groups

def impute_by_station(df, use_global_hourly_stats=None):
    """
    Imputa un DataFrame de una sola estación aplicando los métodos en orden:
    1. Gaps cortos (≤6h) → interpolación lineal.
    2. Gaps medios (6-72h) → media/mediana por hora (estacional).
    3. Gaps largos (≥72h) → se dejan para KNN (se aplica después externamente).

    Si use_global_hourly_stats es un dict (hour -> valor), se usan esas medias globales;
    de lo contrario, se calculan a partir de los propios datos de la estación.
    """
    df_imp = df.copy()
    # Procesar cada columna numérica
    for col in NUM_COLS:
        if col not in df_imp.columns:
            continue
        serie = df_imp[col]
        if serie.isna().sum() == 0:
            continue

        # Obtener gaps
        gaps = get_gap_lengths(serie)
        for idx_tuple, L in gaps.items():
            idx = pd.Index(idx_tuple)  # convertir a Index
            if L <= SHORT_GAP_MAX:
                # Interpolación lineal
                # interpolate con limit_direction='both' rellena usando valores anteriores y posteriores
                serie_interp = serie.interpolate(method='linear', limit_direction='both')
                df_imp.loc[idx, col] = serie_interp.loc[idx]
            elif L < MEDIUM_GAP_MAX:
                # Imputación estacional por hora
                if use_global_hourly_stats is not None:
                    # Usar estadísticos globales
                    hourly_vals = idx.hour.map(use_global_hourly_stats)
                else:
                    # Calcular estadísticos por hora de la propia estación
                    # Excluimos los valores del gap actual (aunque son NaN, no influyen)
                    if SEASONAL_STAT == 'mean':
                        hourly_stats = serie.dropna().groupby(serie.dropna().index.hour).mean()
                    else:
                        hourly_stats = serie.dropna().groupby(serie.dropna().index.hour).median()
                    hourly_vals = idx.hour.map(hourly_stats)
                df_imp.loc[idx, col] = hourly_vals.values
            else:
                # Gaps largos: se dejarán para KNN (no hacemos nada aquí)
                pass
    return df_imp

def apply_knn_imputation(df, num_cols, extra_cols=None):
    """
    Aplica KNNImputer a las columnas numéricas del DataFrame.
    extra_cols: columnas adicionales que se incluirán como predictores pero no se imputarán (ej. station_id).
    """
    cols_to_impute = [c for c in num_cols if c in df.columns]
    if extra_cols:
        all_cols = cols_to_impute + extra_cols
    else:
        all_cols = cols_to_impute

    # Seleccionar subconjunto
    sub = df[all_cols].copy()
    # KNNImputer solo funciona con datos numéricos; ya lo son.
    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
    imputed_array = imputer.fit_transform(sub)
    # Reasignar solo las columnas imputadas (las extra no se modifican)
    df_imp = df.copy()
    df_imp[cols_to_impute] = imputed_array[:, :len(cols_to_impute)]
    return df_imp

# ============================================================================
# PROCESAMIENTO POR ESTACIÓN (INDIVIDUAL)
# ============================================================================

def process_by_station():
    print("\n=== Imputación por estación (individual) ===")
    # Buscar todos los archivos _outliers.csv en INPUT_DIR
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        print(f"Procesando {station_name} ...")
        df = load_station_data(filepath)
        # Imputar con estadísticos propios
        df_imp = impute_by_station(df, use_global_hourly_stats=None)
        # Aplicar KNN a los NaN restantes (gaps largos)
        # Nota: KNN se aplica a todas las filas, pero solo modificará NaN
        df_imp = apply_knn_imputation(df_imp, NUM_COLS)
        # Renombrar O3_for_impute a O3 para los modelos
        if 'O3_for_impute' in df_imp.columns:
            df_imp.rename(columns={'O3_for_impute': 'O3'}, inplace=True)
        # Guardar
        out_path = os.path.join(OUTPUT_BY_STATION, f"{station_name}.csv")
        df_imp.to_csv(out_path, index=True)
        print(f"  Guardado en {out_path}")

# ============================================================================
# PROCESAMIENTO GLOBAL (TODAS LAS ESTACIONES JUNTAS)
# ============================================================================

def process_global():
    print("\n=== Imputación global (todas las estaciones) ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    # Lista para concatenar
    dfs = []
    station_names = []
    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        station_names.append(station_name)
        df = load_station_data(filepath)
        df['station_id'] = station_name  # identificador de estación
        dfs.append(df)

    # Concatenar todos
    df_all = pd.concat(dfs, axis=0)
    # Ordenar por datetime y station_id? No es necesario, pero para KNN puede ayudar
    # Dejamos como está.

    # 1. Calcular estadísticos horarios globales (media/mediana por hora) para cada variable
    #    usando todos los datos no NaN de todas las estaciones.
    global_hourly_stats = {}
    for col in NUM_COLS:
        if col not in df_all.columns:
            continue
        s = df_all[col].dropna()
        if SEASONAL_STAT == 'mean':
            hourly = s.groupby(s.index.hour).mean()
        else:
            hourly = s.groupby(s.index.hour).median()
        global_hourly_stats[col] = hourly

    # 2. Imputar cada estación por separado usando los estadísticos globales
    #    (para interpolación y estacional). Hacemos loop sobre las estaciones.
    df_all_imp = df_all.copy()
    for station in station_names:
        mask = df_all_imp['station_id'] == station
        df_station = df_all_imp.loc[mask].copy()
        # Aplicar imputación con estadísticos globales (pasamos el dict de medias por hora)
        # Necesitamos una versión de impute_by_station que acepte un dict con las medias por columna.
        # Modificamos la función para que acepte un dict de series indexadas por hora.
        # Para simplificar, reimplementamos aquí un bucle específico.
        for col in NUM_COLS:
            if col not in df_station.columns:
                continue
            serie = df_station[col]
            if serie.isna().sum() == 0:
                continue
            gaps = get_gap_lengths(serie)
            for idx_tuple, L in gaps.items():
                idx = pd.Index(idx_tuple)
                if L <= SHORT_GAP_MAX:
                    # Interpolación lineal dentro de la estación (usamos la serie completa de la estación)
                    serie_interp = serie.interpolate(method='linear', limit_direction='both')
                    df_station.loc[idx, col] = serie_interp.loc[idx]
                elif L < MEDIUM_GAP_MAX:
                    # Usar estadístico global por hora
                    hourly_vals = idx.hour.map(global_hourly_stats[col])
                    df_station.loc[idx, col] = hourly_vals.values
                # largos se dejan para KNN
        # Actualizar el DataFrame global
        df_all_imp.loc[mask] = df_station

    # 3. Aplicar KNN imputer a todo el conjunto (incluyendo station_id como predictor)
    #    para los gaps largos restantes.
    #    Incluimos station_id como variable numérica (convertimos a código)
    #    Primero, convertir station_id a código numérico (0,1,2,...)
    df_all_imp['station_code'] = pd.factorize(df_all_imp['station_id'])[0]
    extra_cols = ['station_code']
    # Aplicar KNN
    df_all_imp = apply_knn_imputation(df_all_imp, NUM_COLS, extra_cols=extra_cols)
    # Eliminar station_code (ya no necesario)
    df_all_imp.drop(columns=['station_code'], inplace=True)

    # 4. Separar por estación y guardar
    for station in station_names:
        df_station = df_all_imp[df_all_imp['station_id'] == station].copy()
        df_station.drop(columns=['station_id'], inplace=True)
        # Renombrar O3_for_impute a O3
        if 'O3_for_impute' in df_station.columns:
            df_station.rename(columns={'O3_for_impute': 'O3'}, inplace=True)
        out_path = os.path.join(OUTPUT_GLOBAL, f"{station}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado {station} en {out_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("Iniciando imputación de datos...")
    process_by_station()
    process_global()
    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_BY_STATION}")
    print(f"  - {OUTPUT_GLOBAL}")

# Imputacion valores NAN y OUTLIERS segun rango temporal con metodo por Transecto y metodo General

In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 2 (adaptado a transectos): Imputación de valores NaN según duración del gap.
Ahora se generan dos tipos de datasets:
  - Por transecto: un único archivo por transecto con todas sus estaciones.
  - Global: un archivo por estación (como antes) pero usando datos de todas las estaciones.

Entrada: Archivos *_outliers.csv (de Código 1) en Datos_TFG_outliers/
Salida:
  - imputed_by_transect/ : archivos Transecto_1.csv, Transecto_2.csv, ... (con columna Estacion)
  - imputed_global/      : archivos por estación (igual que antes)
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import KNNImputer

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
INPUT_DIR = BASE_DIR  # donde están los *_outliers.csv

OUTPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
OUTPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_TRANSECT, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

# Umbrales (horas)
SHORT_GAP_MAX = 6
MEDIUM_GAP_MAX = 72
LONG_GAP_MIN = 72

# Método para imputación estacional ('mean' o 'median')
SEASONAL_STAT = 'mean'

# Columnas numéricas a imputar (incluye O3_for_impute)
NUM_COLS = [
    'NO', 'NO2', 'NOx', 'O3_for_impute',
    'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Dist.', 'Angulo'
]

# Para KNN
KNN_NEIGHBORS = 5

# ============================================================================
# FUNCIONES AUXILIARES (comunes)
# ============================================================================

def load_station_data(filepath):
    """Carga un CSV con índice datetime."""
    df = pd.read_csv(filepath, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df

def get_gap_lengths(series):
    """
    Devuelve un diccionario {índice_tupla: longitud} para cada grupo consecutivo de NaN.
    """
    mask = series.isna()
    groups = mask.ne(mask.shift()).cumsum()
    gap_groups = {}
    for g in groups[mask].unique():
        idx = groups[groups == g].index
        gap_groups[tuple(idx)] = len(idx)
    return gap_groups

def impute_dataframe(df, hourly_stats_dict):
    """
    Aplica imputación a un DataFrame (de una sola estación, con índice único):
      - Gaps cortos → interpolación lineal (usando la propia serie).
      - Gaps medios → relleno con estadístico horario (desde hourly_stats_dict).
      - Gaps largos → se dejan para KNN (no se modifican aquí).
    hourly_stats_dict: dict {nombre_columna: Series indexada por hora (0-23)}
    """
    df_imp = df.copy()
    for col in NUM_COLS:
        if col not in df_imp.columns:
            continue
        serie = df_imp[col]
        if serie.isna().sum() == 0:
            continue

        gaps = get_gap_lengths(serie)
        for idx_tuple, L in gaps.items():
            idx = pd.Index(idx_tuple)
            if L <= SHORT_GAP_MAX:
                # Interpolación lineal
                serie_interp = serie.interpolate(method='linear', limit_direction='both')
                df_imp.loc[idx, col] = serie_interp.loc[idx]
            elif L < MEDIUM_GAP_MAX:
                # Imputación estacional por hora
                if col in hourly_stats_dict:
                    hourly_vals = idx.hour.map(hourly_stats_dict[col])
                    # Aseguramos que el índice coincida (idx es el mismo que el de hourly_vals)
                    df_imp.loc[idx, col] = hourly_vals
                # Si no hay estadístico (columna no presente), se deja NaN (lo tomará KNN)
    return df_imp

def apply_knn_imputation(df, num_cols, extra_cols=None):
    """
    Aplica KNNImputer a las columnas num_cols, usando extra_cols como predictores adicionales.
    Retorna df con las columnas imputadas.
    """
    cols_to_impute = [c for c in num_cols if c in df.columns]
    if extra_cols:
        all_cols = cols_to_impute + extra_cols
    else:
        all_cols = cols_to_impute

    sub = df[all_cols].copy()
    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
    imputed_array = imputer.fit_transform(sub)
    df_imp = df.copy()
    df_imp[cols_to_impute] = imputed_array[:, :len(cols_to_impute)]
    return df_imp

def fill_remaining_nans_with_mean(df, cols):
    """
    Rellena los NaN que aún puedan quedar en las columnas especificadas con la media de cada columna.
    Si una columna es completamente NaN, se rellena con 0 (como último recurso).
    """
    for col in cols:
        if col in df.columns and df[col].isna().any():
            mean_val = df[col].mean(skipna=True)
            if pd.isna(mean_val):
                # Toda la columna es NaN -> no se puede calcular media, usamos 0
                mean_val = 0.0
            df[col].fillna(mean_val, inplace=True)
    return df

# ============================================================================
# PROCESAMIENTO POR TRANSECTO (CORREGIDO)
# ============================================================================

def process_by_transect():
    print("\n=== Imputación por transecto (agrupando estaciones) ===")

    # Buscar todos los archivos _outliers.csv
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    # Diccionario para agrupar por transecto: key = nombre del transecto, value = lista de DataFrames
    transect_dict = {}

    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        df = load_station_data(filepath)

        # Obtener el nombre del transecto de la columna 'Transecto' (debe ser constante)
        if 'Transecto' not in df.columns:
            print(f"  Advertencia: {filepath.name} no tiene columna 'Transecto'. Se omite.")
            continue
        transect_names = df['Transecto'].dropna().unique()
        if len(transect_names) == 0:
            print(f"  Advertencia: {filepath.name} no tiene valores en Transecto. Se omite.")
            continue
        transect = transect_names[0]  # tomamos el primero (debería ser único)
        # Limpiar nombre para usarlo como nombre de archivo (reemplazar espacios por _)
        transect_clean = transect.replace(" ", "_")

        # Añadir columna con el nombre de la estación (si no existe, la creamos)
        if 'Estacion' not in df.columns:
            df['Estacion'] = station_name

        # Guardar en el diccionario
        if transect_clean not in transect_dict:
            transect_dict[transect_clean] = []
        transect_dict[transect_clean].append((station_name, df))

    # Procesar cada transecto
    for transect_clean, station_list in transect_dict.items():
        print(f"\nProcesando transecto: {transect_clean}")

        # 1. Concatenar todas las estaciones del transecto para calcular estadísticos horarios
        df_all = pd.concat([df for _, df in station_list], axis=0, sort=False)
        df_all = df_all.sort_index()

        # Calcular estadísticos horarios del transecto para cada columna
        hourly_stats = {}
        for col in NUM_COLS:
            if col not in df_all.columns:
                continue
            s = df_all[col].dropna()
            if SEASONAL_STAT == 'mean':
                hourly = s.groupby(s.index.hour).mean()
            else:
                hourly = s.groupby(s.index.hour).median()
            hourly_stats[col] = hourly

        # 2. Imputar cada estación por separado (para no mezclar estaciones en gaps)
        df_imputed_list = []
        for station_name, df_station in station_list:
            # Aplicar imputación (interpolación y estacional) con estadísticos del transecto
            df_station_imp = impute_dataframe(df_station, hourly_stats)
            df_imputed_list.append(df_station_imp)

        # 3. Concatenar todas las estaciones ya imputadas (parcialmente)
        df_concat = pd.concat(df_imputed_list, axis=0, sort=False)
        df_concat = df_concat.sort_index()

        # 4. Preparar para KNN: convertir Estacion a código numérico
        df_concat['station_code'] = pd.factorize(df_concat['Estacion'])[0]
        extra_cols = ['station_code']

        # 5. Aplicar KNN a los NaN restantes (gaps largos)
        df_concat = apply_knn_imputation(df_concat, NUM_COLS, extra_cols=extra_cols)

        # 5b. Rellenar NaN residuales con la media de la columna
        df_concat = fill_remaining_nans_with_mean(df_concat, NUM_COLS)

        # 6. Eliminar la columna auxiliar station_code
        df_concat.drop(columns=['station_code'], inplace=True)

        # 7. Renombrar O3_for_impute a O3
        if 'O3_for_impute' in df_concat.columns:
            df_concat.rename(columns={'O3_for_impute': 'O3'}, inplace=True)

        # 8. Guardar el DataFrame del transecto completo
        out_path = os.path.join(OUTPUT_BY_TRANSECT, f"{transect_clean}.csv")
        df_concat.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path} (shape: {df_concat.shape})")

# ============================================================================
# PROCESAMIENTO GLOBAL (todas las estaciones juntas)
# ============================================================================

def process_global():
    print("\n=== Imputación global (todas las estaciones) ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    dfs = []
    station_names = []
    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        station_names.append(station_name)
        df = load_station_data(filepath)
        df['station_id'] = station_name
        dfs.append(df)

    df_all = pd.concat(dfs, axis=0)
    # Ordenar por datetime (importante para la imputación)
    df_all = df_all.sort_index()

    # 1. Calcular estadísticos horarios globales
    global_hourly_stats = {}
    for col in NUM_COLS:
        if col not in df_all.columns:
            continue
        s = df_all[col].dropna()
        if SEASONAL_STAT == 'mean':
            hourly = s.groupby(s.index.hour).mean()
        else:
            hourly = s.groupby(s.index.hour).median()
        global_hourly_stats[col] = hourly

    # 2. Imputar por estación (aplicando la función impute_dataframe a cada una por separado,
    #    pero usando estadísticos globales)
    df_all_imp = df_all.copy()
    for station in station_names:
        mask = df_all_imp['station_id'] == station
        df_station = df_all_imp.loc[mask].copy()
        # Aplicar imputación (interpolación + estacional) usando estadísticos globales
        df_station = impute_dataframe(df_station, global_hourly_stats)
        df_all_imp.loc[mask] = df_station

    # 3. KNN global con station_code
    df_all_imp['station_code'] = pd.factorize(df_all_imp['station_id'])[0]
    extra_cols = ['station_code']
    df_all_imp = apply_knn_imputation(df_all_imp, NUM_COLS, extra_cols=extra_cols)

    # 3b. Rellenar NaN residuales con la media de la columna
    df_all_imp = fill_remaining_nans_with_mean(df_all_imp, NUM_COLS)

    df_all_imp.drop(columns=['station_code'], inplace=True)

    # 4. Renombrar O3_for_impute a O3
    if 'O3_for_impute' in df_all_imp.columns:
        df_all_imp.rename(columns={'O3_for_impute': 'O3'}, inplace=True)

    # 5. Separar por estación y guardar
    for station in station_names:
        df_station = df_all_imp[df_all_imp['station_id'] == station].copy()
        df_station.drop(columns=['station_id'], inplace=True)
        out_path = os.path.join(OUTPUT_GLOBAL, f"{station}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("Iniciando imputación de datos (por transecto y global)...")
    process_by_transect()
    process_global()
    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_BY_TRANSECT}")
    print(f"  - {OUTPUT_GLOBAL}")

Iniciando imputación de datos (por transecto y global)...

=== Imputación por transecto (agrupando estaciones) ===

Procesando transecto: Transecto_1
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_by_transect/Transecto_1.csv (shape: (35042, 13))

Procesando transecto: Transecto_2
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_by_transect/Transecto_2.csv (shape: (35042, 13))

=== Imputación global (todas las estaciones) ===
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_global/T1_E1_Alicante.csv
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_global/T1_E2_Elda.csv
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_global/T2_E1_Elche.csv
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_global/T2_E2_Elda.csv

Proceso completado. Revise las carpetas:
  - /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_by_transect
  - /Volumes/copi

In [12]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 2 (adaptado a transectos): Imputación de valores NaN según duración del gap.
Ahora se generan dos tipos de datasets:
  - Por transecto: un único archivo por transecto con todas sus estaciones.
  - Global: un archivo por estación (como antes) pero usando datos de todas las estaciones.

Entrada: Archivos *_outliers.csv (de Código 1) en Datos_TFG_outliers/
Salida:
  - imputed_by_transect/ : archivos Transecto_1.csv, Transecto_2.csv, ... (con columna Estacion)
  - imputed_global/      : archivos por estación (igual que antes)
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import KNNImputer

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
INPUT_DIR = BASE_DIR  # donde están los *_outliers.csv

OUTPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
OUTPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_TRANSECT, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

# Umbrales (horas)
SHORT_GAP_MAX = 6
MEDIUM_GAP_MAX = 72
LONG_GAP_MIN = 72

# Método para imputación estacional ('mean' o 'median')
SEASONAL_STAT = 'mean'

# Columnas numéricas a imputar (incluye O3_for_impute)
NUM_COLS = [
    'NO', 'NO2', 'NOx', 'O3_for_impute',
    'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Dist.', 'Angulo'
]

# Para KNN
KNN_NEIGHBORS = 5

# ============================================================================
# FUNCIONES AUXILIARES (comunes)
# ============================================================================

def load_station_data(filepath):
    """Carga un CSV con índice datetime."""
    df = pd.read_csv(filepath, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df

def get_gap_lengths(series):
    """
    Devuelve un diccionario {índice_tupla: longitud} para cada grupo consecutivo de NaN.
    """
    mask = series.isna()
    groups = mask.ne(mask.shift()).cumsum()
    gap_groups = {}
    for g in groups[mask].unique():
        idx = groups[groups == g].index
        gap_groups[tuple(idx)] = len(idx)
    return gap_groups

def compute_hourly_stats(series, stat='mean'):
    """
    Calcula el estadístico horario (media o mediana) y asegura que existan valores para todas las horas (0-23).
    Si no hay datos para una hora, se rellena con el valor más cercano (forward/backward fill).
    Si la serie está vacía, retorna una Serie de NaN para todas las horas.
    """
    if series.empty:
        return pd.Series(np.nan, index=range(24))
    if stat == 'mean':
        hourly = series.groupby(series.index.hour).mean()
    else:
        hourly = series.groupby(series.index.hour).median()
    # Asegurar que todas las horas estén presentes
    hourly = hourly.reindex(range(24))
    # Rellenar huecos: primero hacia adelante, luego hacia atrás
    hourly = hourly.ffill().bfill()
    # Si aún quedan NaN (serie completamente vacía), se dejan como NaN (ya se manejará después)
    return hourly

def impute_dataframe(df, hourly_stats_dict):
    """
    Aplica imputación a un DataFrame (de una sola estación, con índice único):
      - Gaps cortos → interpolación lineal (usando la propia serie).
      - Gaps medios → relleno con estadístico horario (desde hourly_stats_dict).
      - Gaps largos → se dejan para KNN (no se modifican aquí).
    hourly_stats_dict: dict {nombre_columna: Series indexada por hora (0-23) con valores listos}
    """
    df_imp = df.copy()
    for col in NUM_COLS:
        if col not in df_imp.columns:
            continue
        serie = df_imp[col]
        if serie.isna().sum() == 0:
            continue

        gaps = get_gap_lengths(serie)
        for idx_tuple, L in gaps.items():
            idx = pd.Index(idx_tuple)
            if L <= SHORT_GAP_MAX:
                # Interpolación lineal
                serie_interp = serie.interpolate(method='linear', limit_direction='both')
                df_imp.loc[idx, col] = serie_interp.loc[idx]
            elif L < MEDIUM_GAP_MAX:
                # Imputación estacional por hora
                if col in hourly_stats_dict:
                    hourly_vals = idx.hour.map(hourly_stats_dict[col])
                    df_imp.loc[idx, col] = hourly_vals
                # Si no hay estadístico, se deja NaN (lo tomará KNN)
    return df_imp

def apply_knn_imputation(df, num_cols, extra_cols=None):
    """
    Aplica KNNImputer a las columnas num_cols, usando extra_cols como predictores adicionales.
    Retorna df con las columnas imputadas.
    """
    cols_to_impute = [c for c in num_cols if c in df.columns]
    if extra_cols:
        all_cols = cols_to_impute + extra_cols
    else:
        all_cols = cols_to_impute

    sub = df[all_cols].copy()
    # Asegurar que las columnas extra no tengan NaN (si las hay)
    if extra_cols:
        for col in extra_cols:
            if sub[col].isna().any():
                # Rellenamos con -1 (código para desconocido) para que KNN pueda operar
                sub[col].fillna(-1, inplace=True)

    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
    imputed_array = imputer.fit_transform(sub)
    df_imp = df.copy()
    df_imp[cols_to_impute] = imputed_array[:, :len(cols_to_impute)]
    return df_imp

def fill_remaining_nans_with_mean(df, cols):
    """
    Rellena los NaN que aún puedan quedar en las columnas especificadas con la media de cada columna.
    Si una columna es completamente NaN, se rellena con 0 (como último recurso).
    """
    for col in cols:
        if col in df.columns and df[col].isna().any():
            mean_val = df[col].mean(skipna=True)
            if pd.isna(mean_val):
                mean_val = 0.0
            df[col] = df[col].fillna(mean_val)
    return df

def ensure_no_nans(df, cols):
    """
    Verifica que no queden NaN en las columnas especificadas. Si los hay, los rellena con 0 y muestra advertencia.
    """
    for col in cols:
        if col in df.columns and df[col].isna().any():
            n_nans = df[col].isna().sum()
            print(f"  ADVERTENCIA: {n_nans} NaN residuales en {col} rellenados con 0.")
            df[col] = df[col].fillna(0)
    return df

# ============================================================================
# PROCESAMIENTO POR TRANSECTO (CORREGIDO)
# ============================================================================

def process_by_transect():
    print("\n=== Imputación por transecto (agrupando estaciones) ===")

    # Buscar todos los archivos _outliers.csv
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    # Diccionario para agrupar por transecto: key = nombre del transecto, value = lista de DataFrames
    transect_dict = {}

    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        df = load_station_data(filepath)

        # Obtener el nombre del transecto de la columna 'Transecto' (debe ser constante)
        if 'Transecto' not in df.columns:
            print(f"  Advertencia: {filepath.name} no tiene columna 'Transecto'. Se omite.")
            continue
        transect_names = df['Transecto'].dropna().unique()
        if len(transect_names) == 0:
            print(f"  Advertencia: {filepath.name} no tiene valores en Transecto. Se omite.")
            continue
        transect = transect_names[0]  # tomamos el primero (debería ser único)
        # Limpiar nombre para usarlo como nombre de archivo (reemplazar espacios por _)
        transect_clean = transect.replace(" ", "_")

        # Añadir columna con el nombre de la estación (si no existe, la creamos)
        if 'Estacion' not in df.columns:
            df['Estacion'] = station_name

        # Guardar en el diccionario
        if transect_clean not in transect_dict:
            transect_dict[transect_clean] = []
        transect_dict[transect_clean].append((station_name, df))

    # Procesar cada transecto
    for transect_clean, station_list in transect_dict.items():
        print(f"\nProcesando transecto: {transect_clean}")

        # 1. Concatenar todas las estaciones del transecto para calcular estadísticos horarios
        df_all = pd.concat([df for _, df in station_list], axis=0, sort=False)
        df_all = df_all.sort_index()

        # Calcular estadísticos horarios del transecto para cada columna
        hourly_stats = {}
        for col in NUM_COLS:
            if col not in df_all.columns:
                continue
            s = df_all[col].dropna()
            hourly_stats[col] = compute_hourly_stats(s, SEASONAL_STAT)

        # 2. Imputar cada estación por separado (para no mezclar estaciones en gaps)
        df_imputed_list = []
        for station_name, df_station in station_list:
            # Aplicar imputación (interpolación y estacional) con estadísticos del transecto
            df_station_imp = impute_dataframe(df_station, hourly_stats)
            df_imputed_list.append(df_station_imp)

        # 3. Concatenar todas las estaciones ya imputadas (parcialmente)
        df_concat = pd.concat(df_imputed_list, axis=0, sort=False)
        df_concat = df_concat.sort_index()

        # 4. Preparar para KNN: convertir Estacion a código numérico
        df_concat['station_code'] = pd.factorize(df_concat['Estacion'])[0]
        extra_cols = ['station_code']

        # 5. Aplicar KNN a los NaN restantes (gaps largos)
        df_concat = apply_knn_imputation(df_concat, NUM_COLS, extra_cols=extra_cols)

        # 5b. Rellenar NaN residuales con la media de la columna
        df_concat = fill_remaining_nans_with_mean(df_concat, NUM_COLS)

        # 5c. Verificación final: asegurar que no quede ningún NaN en NUM_COLS
        df_concat = ensure_no_nans(df_concat, NUM_COLS)

        # 6. Eliminar la columna auxiliar station_code
        df_concat.drop(columns=['station_code'], inplace=True)

        # 7. Renombrar O3_for_impute a O3
        if 'O3_for_impute' in df_concat.columns:
            df_concat.rename(columns={'O3_for_impute': 'O3'}, inplace=True)

        # 8. Guardar el DataFrame del transecto completo
        out_path = os.path.join(OUTPUT_BY_TRANSECT, f"{transect_clean}.csv")
        df_concat.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path} (shape: {df_concat.shape})")

# ============================================================================
# PROCESAMIENTO GLOBAL (todas las estaciones juntas)
# ============================================================================

def process_global():
    print("\n=== Imputación global (todas las estaciones) ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    dfs = []
    station_names = []
    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        station_names.append(station_name)
        df = load_station_data(filepath)
        df['station_id'] = station_name
        dfs.append(df)

    df_all = pd.concat(dfs, axis=0)
    # Ordenar por datetime (importante para la imputación)
    df_all = df_all.sort_index()

    # 1. Calcular estadísticos horarios globales
    global_hourly_stats = {}
    for col in NUM_COLS:
        if col not in df_all.columns:
            continue
        s = df_all[col].dropna()
        global_hourly_stats[col] = compute_hourly_stats(s, SEASONAL_STAT)

    # 2. Imputar por estación (aplicando la función impute_dataframe a cada una por separado,
    #    pero usando estadísticos globales)
    df_all_imp = df_all.copy()
    for station in station_names:
        mask = df_all_imp['station_id'] == station
        df_station = df_all_imp.loc[mask].copy()
        # Aplicar imputación (interpolación + estacional) usando estadísticos globales
        df_station = impute_dataframe(df_station, global_hourly_stats)
        df_all_imp.loc[mask] = df_station

    # 3. KNN global con station_code
    df_all_imp['station_code'] = pd.factorize(df_all_imp['station_id'])[0]
    extra_cols = ['station_code']
    df_all_imp = apply_knn_imputation(df_all_imp, NUM_COLS, extra_cols=extra_cols)

    # 3b. Rellenar NaN residuales con la media de la columna
    df_all_imp = fill_remaining_nans_with_mean(df_all_imp, NUM_COLS)

    # 3c. Verificación final: asegurar que no quede ningún NaN en NUM_COLS
    df_all_imp = ensure_no_nans(df_all_imp, NUM_COLS)

    df_all_imp.drop(columns=['station_code'], inplace=True)

    # 4. Renombrar O3_for_impute a O3
    if 'O3_for_impute' in df_all_imp.columns:
        df_all_imp.rename(columns={'O3_for_impute': 'O3'}, inplace=True)

    # 5. Separar por estación y guardar
    for station in station_names:
        df_station = df_all_imp[df_all_imp['station_id'] == station].copy()
        df_station.drop(columns=['station_id'], inplace=True)
        out_path = os.path.join(OUTPUT_GLOBAL, f"{station}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("Iniciando imputación de datos (por transecto y global)...")
    process_by_transect()
    process_global()
    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_BY_TRANSECT}")
    print(f"  - {OUTPUT_GLOBAL}")

Iniciando imputación de datos (por transecto y global)...

=== Imputación por transecto (agrupando estaciones) ===



Procesando transecto: Transecto_1
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_by_transect/Transecto_1.csv (shape: (35042, 13))

Procesando transecto: Transecto_2
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_by_transect/Transecto_2.csv (shape: (35042, 13))

=== Imputación global (todas las estaciones) ===
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_global/T1_E1_Alicante.csv
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_global/T1_E2_Elda.csv
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_global/T2_E1_Elche.csv
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_global/T2_E2_Elda.csv

Proceso completado. Revise las carpetas:
  - /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_by_transect
  - /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_global


In [21]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 2 (adaptado a transectos) - Versión corregida para evitar NaNs residuales.
Corrección principal: se reemplazaron llamadas a fillna(method='ffill') por .ffill().bfill()
Entrada: Archivos *_outliers.csv en INPUT_DIR
Salida:
  - imputed_by_transect/: archivos Transecto_1.csv, Transecto_2.csv, ...
  - imputed_global/     : archivos por estación
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import KNNImputer

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
INPUT_DIR = BASE_DIR  # donde están los *_outliers.csv

OUTPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
OUTPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_TRANSECT, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

# Umbrales (horas)
SHORT_GAP_MAX = 6
MEDIUM_GAP_MAX = 72
LONG_GAP_MIN = 72

# Método para imputación estacional ('mean' o 'median')
SEASONAL_STAT = 'mean'

# Columnas numéricas a imputar (incluye tanto O3_for_impute como O3 por seguridad)
NUM_COLS = [
    'NO', 'NO2', 'NOx', 'O3_for_impute', 'O3',
    'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Dist.', 'Angulo'
]

# Para KNN
KNN_NEIGHBORS = 5

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def load_station_data(filepath):
    """Carga un CSV con índice datetime."""
    df = pd.read_csv(filepath, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df

def get_gap_lengths(series):
    """
    Devuelve un dict {tuple(indice): longitud} para cada grupo consecutivo de NaN en 'series'.
    """
    mask = series.isna()
    if mask.empty:
        return {}
    groups = mask.ne(mask.shift()).cumsum()
    gap_groups = {}
    for g in groups[mask].unique():
        idx = groups[groups == g].index
        gap_groups[tuple(idx)] = len(idx)
    return gap_groups

def compute_hourly_stats(series, stat='mean'):
    """
    Calcula el estadístico horario (media o mediana) y asegura que existan valores para todas las horas (0-23).
    Si no hay datos para una hora, se rellena con el valor más cercano (ffill/bfill).
    Si la serie está vacía, retorna Serie de NaN.
    """
    if series is None or series.empty:
        return pd.Series(np.nan, index=range(24))
    if stat == 'mean':
        hourly = series.groupby(series.index.hour).mean()
    else:
        hourly = series.groupby(series.index.hour).median()
    hourly = hourly.reindex(range(24))
    hourly = hourly.ffill().bfill()
    return hourly

def impute_dataframe(df, hourly_stats_dict):
    """
    Imputa un DataFrame de una estación:
      - Gaps cortos (<= SHORT_GAP_MAX): interpolación lineal.
      - Gaps medios (< MEDIUM_GAP_MAX): imputación por estadístico horario.
      - Gaps largos (>= MEDIUM_GAP_MAX): se dejan para KNN.
    """
    df_imp = df.copy()
    for col in NUM_COLS:
        if col not in df_imp.columns:
            continue
        df_imp[col] = pd.to_numeric(df_imp[col], errors='coerce')
        serie = df_imp[col]
        if serie.isna().sum() == 0:
            continue

        gaps = get_gap_lengths(serie)
        if not gaps:
            continue

        for idx_tuple, L in gaps.items():
            idx = pd.Index(idx_tuple)
            if L <= SHORT_GAP_MAX:
                serie_interp = serie.interpolate(method='linear', limit_direction='both')
                # Asegurar que las posiciones existan antes de asignar
                common_idx = serie_interp.index.intersection(idx)
                if not common_idx.empty:
                    df_imp.loc[common_idx, col] = serie_interp.loc[common_idx]
            elif L < MEDIUM_GAP_MAX:
                if col in hourly_stats_dict:
                    # map horas a valores horarios
                    hours = pd.Index(idx).hour
                    hourly_series = hourly_stats_dict[col]
                    # si hourly_series es una serie con índice 0-23, indexarlo
                    vals = [hourly_series.get(h, np.nan) if hasattr(hourly_series, "get") else hourly_series.loc[h] for h in hours]
                    df_imp.loc[idx, col] = vals
                # si no hay estadístico, se deja NaN para KNN
            else:
                # gap largo: dejamos NaN para KNN
                pass
    return df_imp

def apply_knn_imputation(df, num_cols, extra_cols=None):
    """
    Aplica KNNImputer a las columnas num_cols (solo las que existan en df),
    usando extra_cols como predictores (deben ser numéricos).
    Retorna df con las columnas imputadas.
    """
    cols_to_impute = [c for c in num_cols if c in df.columns]
    if not cols_to_impute:
        return df

    if extra_cols:
        extra_cols = [c for c in extra_cols if c in df.columns]
        all_cols = cols_to_impute + extra_cols
    else:
        all_cols = cols_to_impute

    sub = df[all_cols].copy()
    for c in sub.columns:
        sub[c] = pd.to_numeric(sub[c], errors='coerce')

    if extra_cols:
        for col in extra_cols:
            if sub[col].isna().any():
                sub[col].fillna(-999, inplace=True)

    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
    try:
        imputed_array = imputer.fit_transform(sub)
    except Exception as e:
        print("  ERROR en KNNImputer:", e)
        print("  Intentando imputación de fallback por media de columna.")
        for c in cols_to_impute:
            mean_val = sub[c].mean(skipna=True)
            sub[c].fillna(mean_val if not pd.isna(mean_val) else 0.0, inplace=True)
        imputed_array = sub.values

    df_imp = df.copy()
    df_imp.loc[:, cols_to_impute] = imputed_array[:, :len(cols_to_impute)]
    return df_imp

def fill_remaining_nans_with_mean(df, cols):
    """
    Rellena los NaN que aún puedan quedar en las columnas especificadas con la media de cada columna.
    Si una columna es completamente NaN, se rellena con 0 (último recurso).
    """
    for col in cols:
        if col in df.columns and df[col].isna().any():
            mean_val = pd.to_numeric(df[col], errors='coerce').mean(skipna=True)
            if pd.isna(mean_val):
                mean_val = 0.0
            df[col] = df[col].fillna(mean_val)
    return df

def ensure_no_nans(df, cols):
    """
    Verifica que no queden NaN en cols; si quedan, rellena con 0 e imprime advertencia.
    """
    for col in cols:
        if col in df.columns and df[col].isna().any():
            n_nans = int(df[col].isna().sum())
            print(f"  ADVERTENCIA: {n_nans} NaN residuales en '{col}' rellenados con 0.")
            df[col] = df[col].fillna(0)
    return df

# ============================================================================
# PROCESAMIENTO POR TRANSECTO (MEJORADO)
# ============================================================================

def process_by_transect():
    print("\n=== Imputación por transecto (agrupando estaciones) ===")

    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    transect_dict = {}

    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        df = load_station_data(filepath)

        if 'O3' in df.columns and 'O3_for_impute' not in df.columns:
            df['O3_for_impute'] = pd.to_numeric(df['O3'], errors='coerce')

        if 'Estacion' not in df.columns:
            df['Estacion'] = station_name
        else:
            df['Estacion'] = df['Estacion'].fillna(station_name)

        if 'Transecto' not in df.columns:
            print(f"  Advertencia: {filepath.name} no tiene columna 'Transecto'. Se omite.")
            continue
        transect_names = df['Transecto'].dropna().unique()
        if len(transect_names) == 0:
            print(f"  Advertencia: {filepath.name} no tiene valores en Transecto. Se omite.")
            continue
        transect = transect_names[0]
        df['Transecto'] = df['Transecto'].fillna(transect)

        transect_clean = str(transect).replace(" ", "_")

        if transect_clean not in transect_dict:
            transect_dict[transect_clean] = []
        transect_dict[transect_clean].append((station_name, df))

    for transect_clean, station_list in transect_dict.items():
        print(f"\nProcesando transecto: {transect_clean}")

        df_all = pd.concat([df for _, df in station_list], axis=0, sort=False)
        df_all = df_all.sort_index()

        hourly_stats = {}
        for col in NUM_COLS:
            if col not in df_all.columns:
                continue
            s = pd.to_numeric(df_all[col], errors='coerce').dropna()
            hourly_stats[col] = compute_hourly_stats(s, SEASONAL_STAT)

        df_imputed_list = []
        for station_name, df_station in station_list:
            df_station['Estacion'] = df_station['Estacion'].fillna(station_name)
            if 'Transecto' in df_station.columns:
                df_station['Transecto'] = df_station['Transecto'].fillna(transect.replace("_", " "))
            for col in NUM_COLS:
                if col in df_station.columns:
                    df_station[col] = pd.to_numeric(df_station[col], errors='coerce')

            df_station_imp = impute_dataframe(df_station, hourly_stats)
            df_imputed_list.append(df_station_imp)

        df_concat = pd.concat(df_imputed_list, axis=0, sort=False)
        df_concat = df_concat.sort_index()

        # Reemplazamos fillna(method=...) por ffill().bfill()
        if 'Estacion' in df_concat.columns:
            df_concat['Estacion'] = df_concat['Estacion'].ffill().bfill().fillna('unknown_station')
        if 'Transecto' in df_concat.columns:
            df_concat['Transecto'] = df_concat['Transecto'].ffill().bfill().fillna(transect_clean.replace("_", " "))

        df_concat['station_code'] = pd.factorize(df_concat['Estacion'])[0]
        extra_cols = ['station_code']

        df_concat = apply_knn_imputation(df_concat, NUM_COLS, extra_cols=extra_cols)
        df_concat = fill_remaining_nans_with_mean(df_concat, NUM_COLS)
        df_concat = ensure_no_nans(df_concat, NUM_COLS)

        if 'station_code' in df_concat.columns:
            df_concat.drop(columns=['station_code'], inplace=True)

        if 'O3_for_impute' in df_concat.columns:
            df_concat['O3'] = df_concat['O3_for_impute']
            df_concat.drop(columns=['O3_for_impute'], inplace=True)

        out_path = os.path.join(OUTPUT_BY_TRANSECT, f"{transect_clean}.csv")
        df_concat.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path} (shape: {df_concat.shape})")

# ============================================================================
# PROCESAMIENTO GLOBAL (MEJORADO)
# ============================================================================

def process_global():
    print("\n=== Imputación global (todas las estaciones) ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    dfs = []
    station_names = []
    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        station_names.append(station_name)
        df = load_station_data(filepath)

        if 'O3' in df.columns and 'O3_for_impute' not in df.columns:
            df['O3_for_impute'] = pd.to_numeric(df['O3'], errors='coerce')

        if 'Estacion' not in df.columns:
            df['Estacion'] = station_name
        else:
            df['Estacion'] = df['Estacion'].fillna(station_name)

        if 'Transecto' in df.columns:
            tran_names = df['Transecto'].dropna().unique()
            if len(tran_names) > 0:
                df['Transecto'] = df['Transecto'].fillna(tran_names[0])

        df['station_id'] = station_name
        dfs.append(df)

    df_all = pd.concat(dfs, axis=0, sort=False)
    df_all = df_all.sort_index()

    global_hourly_stats = {}
    for col in NUM_COLS:
        if col not in df_all.columns:
            continue
        s = pd.to_numeric(df_all[col], errors='coerce').dropna()
        global_hourly_stats[col] = compute_hourly_stats(s, SEASONAL_STAT)

    df_all_imp = df_all.copy()
    for station in station_names:
        mask = df_all_imp['station_id'] == station
        df_station = df_all_imp.loc[mask].copy()
        for col in NUM_COLS:
            if col in df_station.columns:
                df_station[col] = pd.to_numeric(df_station[col], errors='coerce')
        df_station = impute_dataframe(df_station, global_hourly_stats)
        df_all_imp.loc[mask] = df_station

    df_all_imp['station_code'] = pd.factorize(df_all_imp['station_id'])[0]
    extra_cols = ['station_code']
    df_all_imp = apply_knn_imputation(df_all_imp, NUM_COLS, extra_cols=extra_cols)
    df_all_imp = fill_remaining_nans_with_mean(df_all_imp, NUM_COLS)
    df_all_imp = ensure_no_nans(df_all_imp, NUM_COLS)

    if 'station_code' in df_all_imp.columns:
        df_all_imp.drop(columns=['station_code'], inplace=True)

    if 'O3_for_impute' in df_all_imp.columns:
        df_all_imp['O3'] = df_all_imp['O3_for_impute']
        df_all_imp.drop(columns=['O3_for_impute'], inplace=True)

    for station in station_names:
        df_station = df_all_imp[df_all_imp['station_id'] == station].copy()
        if 'station_id' in df_station.columns:
            df_station.drop(columns=['station_id'], inplace=True)
        if 'Estacion' not in df_station.columns or df_station['Estacion'].isna().any():
            df_station['Estacion'] = df_station['Estacion'].ffill().bfill().fillna(station)
        if 'Transecto' in df_station.columns:
            df_station['Transecto'] = df_station['Transecto'].ffill().bfill().fillna('unknown_transect')

        out_path = os.path.join(OUTPUT_GLOBAL, f"{station}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("Iniciando imputación de datos (por transecto y global)...")
    process_by_transect()
    process_global()
    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_BY_TRANSECT}")
    print(f"  - {OUTPUT_GLOBAL}")

Iniciando imputación de datos (por transecto y global)...

=== Imputación por transecto (agrupando estaciones) ===

Procesando transecto: Transecto_1
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_by_transect/Transecto_1.csv (shape: (35042, 12))

Procesando transecto: Transecto_2
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_by_transect/Transecto_2.csv (shape: (35042, 12))

=== Imputación global (todas las estaciones) ===
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_global/T1_E1_Alicante.csv
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_global/T1_E2_Elda.csv
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_global/T2_E1_Elche.csv
  Guardado: /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_global/T2_E2_Elda.csv

Proceso completado. Revise las carpetas:
  - /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_by_transect
  - /Volumes/copi

In [22]:
import pandas as pd
import os

base = "/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_by_transect"
for f in os.listdir(base):
    if f.endswith('.csv'):
        df = pd.read_csv(os.path.join(base, f), index_col=0)
        if df.isna().any().any():
            print(f"⚠️ {f} tiene {df.isna().sum().sum()} NaNs")
        else:
            print(f"✅ {f} sin NaNs")

✅ Transecto_1.csv sin NaNs
✅ Transecto_2.csv sin NaNs


In [23]:
import pandas as pd
import os

base = "/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/imputed_by_transect"
for f in os.listdir(base):
    if f.endswith('.csv'):
        df = pd.read_csv(os.path.join(base, f), index_col=0)
        nan_cols = df.columns[df.isna().any()].tolist()
        print(f"\n{f}: {len(nan_cols)} columnas con NaN")
        for col in nan_cols:
            n_nan = df[col].isna().sum()
            print(f"  - {col}: {n_nan} NaN")


Transecto_1.csv: 0 columnas con NaN

Transecto_2.csv: 0 columnas con NaN
